
DELTA LIVE TABLES

In [0]:
%sql


CREATE OR REFRESH STREAMING TABLE orders_bronze AS
SELECT * FROM STREAM READ_FILES(
    '/Volumes/capstone_catalog_new/captone_schema_new/dinedash_data/orders/',
    format => 'json'

);


message
The operation was successfully executed.


In [0]:
%sql

CREATE OR REFRESH MATERIALIZED VIEW restaurants_bronze AS 
SELECT  * FROM read_files(
    '/Volumes/capstone_catalog_new/captone_schema_new/dinedash_data/dim_restaurants.csv',

    format => 'csv' ,
    inferschema => true,
    header => true
)


result
The operation was successfully executed.


In [0]:
%sql

CREATE OR REFRESH MATERIALIZED VIEW items_bronze AS 
SELECT  * FROM read_files(
    '/Volumes/capstone_catalog_new/captone_schema_new/dinedash_data/dim_menu_items.csv',

    format => 'csv' ,
    inferschema => true,
    header => true
);


result
The operation was successfully executed.


In [0]:
%sql

CREATE OR REFRESH MATERIALIZED VIEW customers_bronze AS 
SELECT  * FROM read_files(
    '/Volumes/capstone_catalog_new/captone_schema_new/dinedash_data/dim_customers.csv',
    format => 'csv' ,
    header => true
); 

result
The operation was successfully executed.


In [0]:
%sql

CREATE OR REFRESH MATERIALIZED VIEW agents_bronze AS
SELECT * FROM READ_FILES(
    '/Volumes/capstone_catalog_new/captone_schema_new/dinedash_data/dim_delivery_agents.csv',

    format => 'csv' ,
    inferschema => true,
    header => true

);

result
The operation was successfully executed.


In [0]:
%sql

CREATE OR REFRESH MATERIALIZED VIEW  locations_bronze AS
SELECT * FROM READ_FILES(
    '/Volumes/capstone_catalog_new/captone_schema_new/dinedash_data/dim_locations.csv',

    format => 'csv' ,
    inferschema => true,
    header => true

)

result
The operation was successfully executed.


SILVER LAYER(Clean and Normalize)

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE cleaned_orders(
    CONSTRAINT valid_order_id EXPECT(order_id IS NOT NULL)ON VIOLATION DROP ROW,
    CONSTRAINT valid_customer_id EXPECT(customer_id IS NOT NULL)ON VIOLATION DROP ROW,
    CONSTRAINT valid_items EXPECT(size(items_ordered)> 0)ON VIOLATION DROP ROW

)
AS
SELECT 
    order_id,
    timestamp as orders_ts,
    customer_id,
    restaurant_id,
    agent_id,
    delivery_location_id,
    items_ordered,
    explode(items_ordered) AS item,
    total_amount,
    tip,
    payment_method,
    order_status
FROM STREAM(orders_bronze);    



message
The operation was successfully executed.


In [0]:
%sql


CREATE OR REFRESH STREAMING TABLE orders_silver
(
    CONSTRAINT valid_restaurant_name EXPECT(restaurant_name IS NOT NULL) ON VIOLATION DROP ROW,
    CONSTRAINT valid_agent EXPECT(agent_name IS NOT NULL) ON VIOLATION DROP ROW,
    CONSTRAINT valid_item EXPECT(item_id IS NOT NULL) ON VIOLATION DROP ROW
)
AS
SELECT
  o.order_id,
  o.orders_ts,
  o.customer_id,
  c.name AS customer_name,
  c.email,
  o.restaurant_id,
  r.name AS restaurant_name,
  r.cuisines,
  r.rating AS restaurant_rating,
  o.agent_id,
  a.name AS agent_name,
  a.rating AS agent_rating,
  o.delivery_location_id,
  l.city,
  o.item.item_id AS item_id,
  i.item_name,
  i.category,
  o.item.price AS item_price,
  o.item.quantity AS item_quantity,
  o.total_amount,
  o.tip,
  o.payment_method,
  o.order_status
FROM STREAM(Live.cleaned_orders) o
LEFT JOIN Live.customers_bronze c ON o.customer_id = c.customer_id
LEFT JOIN Live.restaurants_bronze r ON o.restaurant_id = r.restaurant_id
LEFT JOIN Live.agents_bronze a ON o.agent_id = a.agent_id
LEFT JOIN Live.items_bronze i ON o.item.item_id = i.item_id
LEFT JOIN Live.locations_bronze l ON o.delivery_location_id = l.location_id;

message
The operation was successfully executed.


In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE orders_with_agents AS
SELECT 
  o.order_id,
  o.orders_ts,
  o.agent_id,
  a.name AS agent_name,
  a.rating AS agent_rating,
  o.tiP
  FROM STREAM(Live.cleaned_orders)O
  LEFT JOIN agents_bronze a
  ON o.agent_id = a.agent_id;

message
The operation was successfully executed.


In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE orders_with_items AS
SELECT
  o.order_id,
  o.order_ts,
  exploded.item_id,
  i.item_name,
  i.category,
  exploded.price,
  exploded.quantity
FROM (
    SELECT order_id,timestamp as order_ts,restaurant_id,explode(items_ordered) AS exploded
    FROM STREAM(Live.orders_bronze)
)O
LEFT JOIN items_bronze i
ON o.exploded.item_id = i.item_id;

message
The operation was successfully executed.


In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE orders_with_restaurants AS
SELECT
  o.order_id,
  o.orders_ts,
  o.delivery_location_id,
  l.city,
  l.state,
  l.area
FROM STREAM(Live.cleaned_orders)o
LEFT JOIN locations_bronze l
ON o.delivery_location_id = l.location_id;  

message
The operation was successfully executed.


GOLD LAYER(Insightful Metrics)

In [0]:
%sql

 CREATE OR REFRESH MATERIALIZED VIEW top_selling_items AS
 SELECT item_name,SUM(item_quantity) AS total_qty_sold
 FROM orders_silver
 GROUP BY item_name
 ORDER BY total_qty_sold DESC
 LIMIT 10;


result
The operation was successfully executed.


In [0]:
%sql

CREATE OR REFRESH MATERIALIZED VIEW total_orders_per_restaurant AS
SELECT restaurant_name,COUNT(DISTINCT order_id)AS total_orders
FROM orders_silver
GROUP BY restaurant_name;

result
The operation was successfully executed.


In [0]:
%sql

CREATE OR REFRESH MATERIALIZED VIEW avg_fees_by_city AS SELECT city,AVG(restaurant_rating) AS avg_restaurant_rating,AVG(tip) AS avg_tip
FROM orders_silver
GROUP BY city;

result
The operation was successfully executed.


In [0]:
%sql


CREATE OR REFRESH MATERIALIZED VIEW agent_order_count AS
SELECT agent_name,COUNT(DISTINCT order_id)AS orders_handles,AVG(agent_rating)AS avg_rating
FROM orders_silver
GROUP BY agent_name
ORDER BY orders_handles DESC;

result
The operation was successfully executed.


In [0]:
%sql



CREATE OR REFRESH MATERIALIZED VIEW daily_revenue_trend AS SELECT DATE(orders_ts)AS order_date,SUM(total_amount)AS total_revenue
FROM Live.orders_silver
GROUP BY order_date
ORDER BY order_date;

result
The operation was successfully executed.


In [0]:
%sql


CREATE OR REFRESH MATERIALIZED VIEW customers_ltv AS SELECT customer_id,customer_name,SUM(total_amount)AS lifetime_spend
FROM orders_silver
GROUP BY customer_id,customer_name
ORDER BY lifetime_spend DESC;

    

result
The operation was successfully executed.


In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE orders_with_customers AS 
SELECT 
   o.order_id,
   o.customer_id,
   c.name AS customer_name,
   c.location_id AS customer_location_id,
   o.total_amount,
   o.tip,
   o.payment_method,
   o.order_status
FROM STREAM(Live.cleaned_orders) o
LEFT JOIN Live.customers_bronze c
ON o.customer_id = c.customer_id;

message
The operation was successfully executed.


In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW customer_order_count AS SELECT customer_name,COUNT(*)AS total_orders 
FROM orders_with_customers
GROUP BY customer_name;

result
The operation was successfully executed.
